# Colab SFT: Llama 3.1 8B (O-RAN Policy Decomposition)

This notebook performs supervised fine-tuning (SFT) with QLoRA on free-tier T4.

Configuration implemented:
- Input: intent.raw_text only
- Output: strict policy_output JSON
- Split: 90/10
- Upload target: Hugging Face model repo
- Includes automatic structural validator

In [9]:
!pip -q install -U transformers datasets accelerate peft trl bitsandbytes huggingface_hub sentencepiece

In [10]:
from huggingface_hub import login
login()  # This will prompt for your HF token

In [11]:
from huggingface_hub import login
login()
import json
import torch
from typing import Any, Dict, List, Tuple

from datasets import load_dataset
from huggingface_hub import login, create_repo
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, set_seed
from trl import SFTTrainer

# Use a public Mistral model for Colab/T4 compatibility.
# You can change this to a local Drive path if you prefer a locally-stored model.
MODEL_ID = "mistralai/mistral-7b-v0.1"  # public Mistral 7B variant

HF_DATASET_REPO = "HikkenNoAce/Intent_Decomposition_to_Sub_Intents_for_O_RAN_networks"
USE_HF_DATASET_REPO = True
LOCAL_JSON_PATH = "/content/dataset.json"

HF_OUTPUT_REPO = "YOUR_USERNAME/llama31-8b-oran-sft-lora"
PUSH_TO_HUB = True
PRIVATE_MODEL_REPO = True

SEED = 42
TEST_SIZE = 0.10
MAX_SEQ_LEN = 1024
NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
GRAD_ACCUM = 8
OUTPUT_DIR = "/content/mistral7b_oran_sft"

SYSTEM_PROMPT = (
    "You are an O-RAN policy decomposition engine. Given a raw intent text, "
    "output ONLY a valid JSON object for policy_output with exactly these top-level keys: "
    "SMO, RIC, CU, DU, RU. Do not add explanations, markdown, or extra keys."
 )

set_seed(SEED)

# Attempt to use Colab secret HF_TOKEN if available
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        login(token=hf_token)
        print("Logged in with HF_TOKEN secret")
    else:
        print("HF_TOKEN secret not found. Public access still works.")
except Exception:
    print("Not in Colab secret context. Call login() manually if needed.")

Not in Colab secret context. Call login() manually if needed.


In [ ]:
# Load dataset + prepare chat-format records (robust loader with fallbacks)
from huggingface_hub import hf_hub_download
import os

def load_train_split_from_repo(repo_id: str, local_json_path: str):
    print(f"Attempting to load dataset directly from HF dataset repo: {repo_id}")
    try:
        loaded = load_dataset(repo_id)
        split_name = "train" if "train" in loaded else list(loaded.keys())[0]
        print(f"Loaded split '{split_name}' from repo: {repo_id}")
        return loaded[split_name]
    except Exception as e:
        print("Direct HF dataset load failed:", e)
        print("Trying to download raw dataset.json from HF dataset repo...")
        try:
            dataset_file = hf_hub_download(
                repo_id=repo_id,
                filename="dataset.json",
                repo_type="dataset",
                token=True,
            )
            print("Downloaded dataset.json from HF repo to:", dataset_file)
            return load_dataset("json", data_files=dataset_file, split="train")
        except Exception as e2:
            print("Download from HF repo failed:", e2)
            if os.path.exists(local_json_path):
                print("Falling back to local JSON path:", local_json_path)
                return load_dataset("json", data_files=local_json_path, split="train")
            raise RuntimeError("Unable to load dataset from HF repo or local path") from e2

# Use the robust loader
if USE_HF_DATASET_REPO:
    raw_dataset = load_train_split_from_repo(HF_DATASET_REPO, LOCAL_JSON_PATH)
else:
    raw_dataset = load_dataset("json", data_files=LOCAL_JSON_PATH, split="train")
    print(f"Loaded local JSON: {LOCAL_JSON_PATH}")

print(raw_dataset)
print("Columns:", raw_dataset.column_names)
print("Sample keys:", list(raw_dataset[0].keys()))

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if "scenario_type" in raw_dataset.column_names:
    encoded = raw_dataset.class_encode_column("scenario_type")
    split = encoded.train_test_split(
        test_size=TEST_SIZE,
        seed=SEED,
        stratify_by_column="scenario_type",
    )
else:
    split = raw_dataset.train_test_split(test_size=TEST_SIZE, seed=SEED)

train_raw = split["train"]
eval_raw = split["test"]

def to_sft_record(example: Dict[str, Any]) -> Dict[str, str]:
    user_text = example["intent"]["raw_text"]
    assistant_text = json.dumps(example["policy_output"], separators=(",", ":"), ensure_ascii=False)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_text},
        {"role": "assistant", "content": assistant_text},
    ]
    # tokenizer.apply_chat_template may not be available for all tokenizers; if it fails, use manual concat
    try:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    except Exception:
        # simple chat-style concat as fallback
        text = SYSTEM_PROMPT + "\nUser: " + user_text + "\nAssistant: " + assistant_text
    return {
        "text": text,
        "raw_text": user_text,
        "policy_output_str": assistant_text,
        "scenario_type": str(example.get("scenario_type", "unknown")),
    }

train_dataset = train_raw.map(to_sft_record)
eval_dataset = eval_raw.map(to_sft_record)

keep_cols = ["text", "raw_text", "policy_output_str", "scenario_type"]
train_dataset = train_dataset.remove_columns([c for c in train_dataset.column_names if c not in keep_cols])
eval_dataset = eval_dataset.remove_columns([c for c in eval_dataset.column_names if c not in keep_cols])

print("Train size:", len(train_dataset))
print("Eval size:", len(eval_dataset))

In [ ]:
# QLoRA setup + SFT training
import gc
import os

gc.collect()
torch.cuda.empty_cache()

# Force fp16 to avoid bf16 AMP issues on T4
os.environ["ACCELERATE_MIXED_PRECISION"] = "fp16"
torch.set_default_dtype(torch.float16)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
 )

# Force full model onto GPU to avoid CPU/disk dispatch on T4
device_map = {"": 0}
max_memory = {"cuda:0": "15GiB"}

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map=device_map,
    max_memory=max_memory,
    dtype=torch.float16,
    low_cpu_mem_usage=True,
 )
model.config.use_cache = False

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
 )

# Transformers >= 4.41 renamed evaluation_strategy -> eval_strategy
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    optim="paged_adamw_8bit",
    fp16=True,
    bf16=False,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=100,
    logging_steps=10,
    save_total_limit=2,
    report_to="none",
    lr_scheduler_type="cosine",
    warmup_steps=100,
    seed=SEED,
 )

# Use formatting_func for TRL versions without dataset_text_field
def format_sft_example(example: Dict[str, Any]) -> str:
    return example["text"]

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    formatting_func=format_sft_example,
 )

train_result = trainer.train()
eval_metrics = trainer.evaluate()
print(train_result.metrics)
print(eval_metrics)

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved locally at {OUTPUT_DIR}")

if PUSH_TO_HUB:
    create_repo(HF_OUTPUT_REPO, repo_type="model", private=PRIVATE_MODEL_REPO, exist_ok=True)
    trainer.model.push_to_hub(HF_OUTPUT_REPO)
    tokenizer.push_to_hub(HF_OUTPUT_REPO)
    print(f"Pushed to {HF_OUTPUT_REPO}")

In [ ]:
# Automatic structural validator + quick evaluation
def extract_first_json(text: str):
    decoder = json.JSONDecoder()
    for i, ch in enumerate(text):
        if ch != "{":
            continue
        try:
            obj, _ = decoder.raw_decode(text[i:])
            if isinstance(obj, dict):
                return obj
        except json.JSONDecodeError:
            pass
    return None

def validate_policy_output_structure(obj: Dict[str, Any]) -> Tuple[bool, List[str]]:
    errors: List[str] = []
    if not isinstance(obj, dict):
        return False, ["Output is not a JSON object"]

    required_top = {"SMO", "RIC", "CU", "DU", "RU"}
    if set(obj.keys()) != required_top:
        errors.append("Top-level keys mismatch")

    slices = {"URLLC", "eMBB", "IoT"}

    try:
        if set(obj["SMO"]["slice_priority"].keys()) != slices:
            errors.append("SMO.slice_priority keys mismatch")
    except Exception:
        errors.append("Invalid SMO structure")

    try:
        prb = obj["RIC"]["prb_allocation"]
        if set(prb.keys()) != slices:
            errors.append("RIC.prb_allocation keys mismatch")
        if any(not isinstance(v, int) for v in prb.values()):
            errors.append("RIC.prb_allocation values must be int")
        if any(v < 5 for v in prb.values()):
            errors.append("RIC.prb_allocation min PRB must be >= 5")
    except Exception:
        errors.append("Invalid RIC structure")

    try:
        du = obj["DU"]
        if du.get("scheduler") not in {"priority", "proportional_fair"}:
            errors.append("DU.scheduler invalid")
        if set(du["queue_weights"].keys()) != slices:
            errors.append("DU.queue_weights keys mismatch")
    except Exception:
        errors.append("Invalid DU structure")

    try:
        if not isinstance(obj["CU"].get("handover_mode"), str):
            errors.append("CU.handover_mode must be string")
        if not isinstance(obj["CU"].get("bearer_priority"), str):
            errors.append("CU.bearer_priority must be string")
        if not isinstance(obj["RU"].get("power_bias"), str):
            errors.append("RU.power_bias must be string")
    except Exception:
        errors.append("Invalid CU/RU structure")

    return len(errors) == 0, errors

def infer_policy(raw_text: str, max_new_tokens: int = 400) -> str:
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": raw_text},
    ]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(trainer.model.device)
    with torch.no_grad():
        outputs = trainer.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def run_structural_validation(eval_ds, n_samples: int = 50):
    n = min(n_samples, len(eval_ds))
    valid_json = 0
    valid_schema = 0
    rows = []

    for i in range(n):
        raw_text = eval_ds[i]["raw_text"]
        pred = infer_policy(raw_text)
        obj = extract_first_json(pred)

        if obj is not None:
            valid_json += 1
            ok, errors = validate_policy_output_structure(obj)
        else:
            ok, errors = False, ["No JSON object found"]

        if ok:
            valid_schema += 1

        rows.append({
            "index": i,
            "scenario_type": eval_ds[i]["scenario_type"],
            "valid_json": obj is not None,
            "valid_schema": ok,
            "errors": errors,
            "prediction_preview": pred[:350],
        })

    print(f"Valid JSON rate: {valid_json / n:.2%}")
    print(f"Schema valid rate: {valid_schema / n:.2%}")
    return rows

validation_rows = run_structural_validation(eval_dataset, n_samples=50)
validation_rows[:3]